# `json2vec` Hello World

This notebook trains the smallest useful JSON2Vec model: two numeric Iris measurements predict the Iris species. The point is not accuracy. The point is to show the complete loop from records, to schema, to training, to prediction and embeddings. This example is intentionally flat; the next tutorials show why arrays matter.

Start with the normal training dependencies plus the bundled Iris JSONL buffer. The examples remove notebook logging noise so the rendered docs stay focused on model behavior.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()


Load a tiny balanced slice of Iris rows. The schema field names match the DataFrame columns, so JSON2Vec can infer the request queries.


In [2]:
records = pl.read_ndjson("docs/data/iris.jsonl").head(36)

records.head()


sepal_length,sepal_width,petal_length,petal_width,species
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""setosa"""
7.0,3.2,4.7,1.4,"""versicolor"""
6.3,3.3,6.0,2.5,"""virginica"""
4.9,3.0,1.4,0.2,"""setosa"""
6.4,3.2,4.5,1.5,"""versicolor"""


The schema declares exactly what the model should read. `Number` fields become numeric tensorfields, and the `Category` field is a supervised target because `target=True` hides it from the input and asks the model to decode it.

In [3]:
model = j2v.Model.from_schema(
    j2v.Number("sepal_length"),
    j2v.Number("petal_length"),
    j2v.Category("species", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.PolarsDataModule(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    sample_rate=1.0,
)


Train for one deliberately small epoch. The tutorials keep batch and epoch counts hardcoded so the example remains quick to run in documentation builds.

In [4]:

trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)


GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.


`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=1` reached.


Prediction uses the same nested batch shape as training: each outer item is one observation, and each observation contains one record.

In [5]:
batch = [[record] for record in records.to_dicts()[:3]]

In [6]:
pprint(model.predict(batch))

{
│   'record/species': {
│   │   'state': {
│   │   │   'valued': [0.41496121883392334, 0.41997310519218445, 0.42189911007881165],
│   │   │   'null': [0.11284373700618744, 0.11292822659015656, 0.11358624696731567],
│   │   │   'padded': [0.25860562920570374, 0.24933700263500214, 0.24542668461799622],
│   │   │   'masked': [0.03981849551200867, 0.04195206239819527, 0.043254680931568146],
│   │   │   'other': [0.17377105355262756, 0.1758095920085907, 0.17583316564559937]
│   │   },
│   │   'content': {
│   │   │   'value': ['setosa', 'setosa', 'setosa'],
│   │   │   'probability': [0.43367207050323486, 0.40919387340545654, 0.4167783260345459],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'setosa', 'probability': 0.43367207050323486},
│   │   │   │   │   {'label': 'versicolor', 'probability': 0.3849213421344757}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'setosa', 'probability': 0.40919387340545654},
│   │   │   │   │   {'label': 'versicolor', 'probability': 0.4016188681125641}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'setosa', 'probability': 0.4167783260345459},
│   │   │   │   │   {'label': 'versicolor', 'probability': 0.3957432210445404}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

Embeddings are opt-in. Passing `embed=True` when constructing the model exposes a root `record` vector for each input observation without changing the schema fields themselves.


In [7]:
pprint(model.embed(batch))


{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.0008178532589226961,
│   │   │   │   0.09154872596263885,
│   │   │   │   -0.1050301045179367,
│   │   │   │   -0.14217326045036316,
│   │   │   │   0.077469602227211,
│   │   │   │   0.3565689027309418,
│   │   │   │   0.5213949680328369,
│   │   │   │   -0.21227335929870605,
│   │   │   │   -0.3102748394012451,
│   │   │   │   -0.28951171040534973,
│   │   │   │   0.016506904736161232,
│   │   │   │   0.39509135484695435,
│   │   │   │   -0.22513850033283234,
│   │   │   │   0.018967576324939728,
│   │   │   │   0.14859269559383392,
│   │   │   │   -0.3173859119415283
│   │   │   ],
│   │   │   [
│   │   │   │   0.007094825617969036,
│   │   │   │   0.08297567814588547,
│   │   │   │   -0.10019178688526154,
│   │   │   │   -0.16869567334651947,
│   │   │   │   0.16423839330673218,
│   │   │   │   0.3420003354549408,
│   │   │   │   0.5197965502738953,
│   │   │   │   -0.20316548645496368,
│   │   │   │   -0.36033502221107483,
│   │   │   │   -0.2416379600763321,
│   │   │   │   -0.012279113754630089,
│   │   │   │   0.39493241906166077,
│   │   │   │   -0.22227880358695984,
│   │   │   │   0.03387381136417389,
│   │   │   │   0.09701269119977951,
│   │   │   │   -0.3079701066017151
│   │   │   ],
│   │   │   [
│   │   │   │   0.01371778268367052,
│   │   │   │   0.07569615542888641,
│   │   │   │   -0.1264125108718872,
│   │   │   │   -0.15437786281108856,
│   │   │   │   0.10631167888641357,
│   │   │   │   0.32418933510780334,
│   │   │   │   0.5456526875495911,
│   │   │   │   -0.1712942123413086,
│   │   │   │   -0.32398200035095215,
│   │   │   │   -0.28830045461654663,
│   │   │   │   -0.02496117353439331,
│   │   │   │   0.4061838686466217,
│   │   │   │   -0.23571787774562836,
│   │   │   │   0.03285170719027519,
│   │   │   │   0.13255921006202698,
│   │   │   │   -0.287898987531662
│   │   │   ]
│   │   ]
│   }
}

The plot is the quickest way to verify what was built: array nodes, tensorfield nodes, targets, embeddings, and parameter counts all appear in the same tree.

In [8]:
model.plot()

Schema
record [array] embed
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=8  parameters=18,153  arrays=1  
fields=3  targets=1  embeds=1  embed=True  n_linear=1
├── sepal_length [number] 3,943 params
│   record/sepal_length
│   query=[*].sepal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  
│   offset=4
├── petal_length [number] 3,943 params
│   record/petal_length
│   query=[*].petal_length  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  
│   offset=4
└── species [category] target 3,659 params
    record/species
    query=[*].species  pooling=query  max_vocab_size=4  topk=[2]  weight=1.0  n_heads=4  p_prune=1.0  n_linear=1  
    n_bands=8  p_unavailable=0.01
